In [4]:
# --- Setup (install if needed) ---
# pip install yfinance tensorflow scikit-learn pandas numpy

import time
import itertools
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ---------- 1) Data: AAPL 5y daily ----------
ticker = "AAPL"
data = yf.download(ticker, period="5y", interval="1d", auto_adjust=True, progress=False)
if data.empty:
    raise ValueError("No data downloaded. Try running again or check your internet/proxy.")

# Ensure DataFrame with a single Close column
close = data[["Close"]].dropna()

# ---------- 2) Build 30-day sliding windows ----------
WINDOW = 30

def make_windows(series_like, window: int):
    """
    Robust window maker:
    - Accepts Series, DataFrame column, or ndarray
    - Flattens to 1D before slicing
    - Returns X: (N, window), y: (N, 1)
    """
    vals = np.asarray(series_like, dtype=np.float32).reshape(-1)  # <-- flatten to 1D
    X, y = [], []
    for i in range(window, len(vals)):
        X.append(vals[i-window:i])  # shape (window,)
        y.append(vals[i])           # scalar
    X = np.asarray(X, dtype=np.float32)             # (N, window)
    y = np.asarray(y, dtype=np.float32).reshape(-1, 1)  # (N, 1)
    return X, y

X_raw, y_raw = make_windows(close["Close"], WINDOW)

# Time-based split: 70% train, 15% val, 15% test
n = len(X_raw)
n_train = int(0.70 * n)
n_val = int(0.15 * n)

X_train_raw, y_train_raw = X_raw[:n_train], y_raw[:n_train]
X_val_raw,   y_val_raw   = X_raw[n_train:n_train+n_val], y_raw[n_train:n_train+n_val]
X_test_raw,  y_test_raw  = X_raw[n_train+n_val:], y_raw[n_train+n_val:]

# ---------- 3) Scale using only train statistics ----------
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

# All are 2D: ok for MinMaxScaler
X_train = x_scaler.fit_transform(X_train_raw)
X_val   = x_scaler.transform(X_val_raw)
X_test  = x_scaler.transform(X_test_raw)

y_train = y_scaler.fit_transform(y_train_raw)
y_val   = y_scaler.transform(y_val_raw)
y_test  = y_scaler.transform(y_test_raw)

# ---------- 4) FNN model builder ----------
def build_fnn(input_dim: int,
              hidden_units: int = 64,
              hidden_layers: int = 2,
              dropout: float = 0.0,
              lr: float = 1e-3) -> keras.Model:
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    for _ in range(hidden_layers):
        model.add(layers.Dense(hidden_units, activation="relu"))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1))  # linear output
    opt = keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss="mse", metrics=["mse"])
    return model

# ---------- 5) Basic hyperparameter tuning (small grid) ----------
param_grid = {
    "hidden_units": [32, 64],
    "hidden_layers": [1, 2, 3],
    "dropout": [0.0, 0.2],
    "lr": [1e-3, 3e-4],
    "batch_size": [32, 64],
}
grid = list(itertools.product(
    param_grid["hidden_units"],
    param_grid["hidden_layers"],
    param_grid["dropout"],
    param_grid["lr"],
    param_grid["batch_size"],
))

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True
)

best = {"val_mse": np.inf, "params": None}
t0 = time.time()

for (hidden_units, hidden_layers, dropout, lr, batch_size) in grid:
    tf.keras.backend.clear_session()
    model = build_fnn(
        input_dim=X_train.shape[1],
        hidden_units=hidden_units,
        hidden_layers=hidden_layers,
        dropout=dropout,
        lr=lr
    )
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=batch_size,
        verbose=0,
        callbacks=[early_stop]
    )
    val_mse = float(np.min(history.history["val_loss"]))
    if val_mse < best["val_mse"]:
        best["val_mse"] = val_mse
        best["params"] = {
            "hidden_units": hidden_units,
            "hidden_layers": hidden_layers,
            "dropout": dropout,
            "lr": lr,
            "batch_size": batch_size
        }

tuning_runtime = time.time() - t0

print("Best hyperparameters (by validation MSE):")
print(best["params"])
print(f"Best validation MSE: {best['val_mse']:.6f}")
print(f"Tuning runtime: {tuning_runtime:.2f} seconds")

# ---------- 6) Retrain on train+val with best params, evaluate on test ----------
X_trval = np.vstack([X_train, X_val])
y_trval = np.vstack([y_train, y_val])

tf.keras.backend.clear_session()
bp = best["params"]
final_model = build_fnn(
    input_dim=X_trval.shape[1],
    hidden_units=bp["hidden_units"],
    hidden_layers=bp["hidden_layers"],
    dropout=bp["dropout"],
    lr=bp["lr"]
)

t1 = time.time()
final_model.fit(
    X_trval, y_trval,
    epochs=200,
    batch_size=bp["batch_size"],
    verbose=0,
    callbacks=[early_stop]
)
train_runtime = time.time() - t1

# Predict and invert scaling
y_pred_scaled = final_model.predict(X_test, verbose=0)
y_pred = y_scaler.inverse_transform(y_pred_scaled)
y_true = y_scaler.inverse_transform(y_test)

# ---------- 7) Metrics ----------
mse = mean_squared_error(y_true, y_pred)
rmse = float(np.sqrt(mse))
mae = mean_absolute_error(y_true, y_pred)

print("\n--- Test Set Performance (Next-Day Close) ---")
print(f"MSE : {mse:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MAE : {mae:.6f}")
print(f"Final training runtime: {train_runtime:.2f} seconds")


Best hyperparameters (by validation MSE):
{'hidden_units': 32, 'hidden_layers': 2, 'dropout': 0.0, 'lr': 0.001, 'batch_size': 32}
Best validation MSE: 0.001894
Tuning runtime: 101.36 seconds


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: loss,mse
  current = self.get_monitor_value(logs)



--- Test Set Performance (Next-Day Close) ---
MSE : 23.637102
RMSE: 4.861800
MAE : 3.398249
Final training runtime: 4.59 seconds
